<a href="https://colab.research.google.com/github/Ehsan-Roohi/DSMC_Python/blob/main/8_Collision_Models_Full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
# --- DSMC Relaxation - V30.0 - MULTI-CELL TEMPORAL CORRELATION ---
import numpy as np
import matplotlib.pyplot as plt
import numba
from numba import njit
import time
import gc
from scipy.special import gammaln

# --- Constants and Physical Parameters ---
MASS_AR = 39.948e-3 / 6.022e23
KB = 1.380649e-23
D_REF_AR = 4.17e-10
T_REF_AR = 273.0
OMEGA_VHS = 0.50
PI = 3.141592654

# --- Simulation Parameters ---
LX = 1.0e-6
RHO_INIT = 1.78
T_INIT = 273.0
NUM_CELLS_X = 100  # Will be adjusted dynamically

# User input for particles per cell will be handled in main execution
PARTICLES_PER_CELL_INIT = 5  # Default value, will be overridden
TOTAL_PARTICLES_SIM = int(NUM_CELLS_X * PARTICLES_PER_CELL_INIT)
N_DENSITY_REAL = RHO_INIT / MASS_AR

# Derived parameters - DT will be calculated dynamically
CELL_VOLUME_CONCEPTUAL = (LX) / NUM_CELLS_X
FNUM = (N_DENSITY_REAL * CELL_VOLUME_CONCEPTUAL) / PARTICLES_PER_CELL_INIT
DT = 1.0e-12  # Will be recalculated
TOTAL_TIME = 4.0e-7
SAMPLING_INTERVAL = 25  # Reduced from 100 to 25 for more frequent sampling

# Missing constants
MAXD = 20  # Maximum deviation for density fluctuation histogram
MTC = 20   # Reduced from 50 to 20 for better statistics
MXC = 10   # Maximum spatial correlation steps
CENTRAL_CELL_IDX = 50  # Central cell index (will be updated during simulation)

# --- Numba-Jitted Core Functions ---
@njit(nopython=True)
def gamma_function_approx(x):
    """
    Corrected implementation of the Gamma function approximation.
    """
    a = 1.0
    y = x
    if y < 1.0:
        a = a / y
        y = y + 1.0
    while y >= 2.0:
        y = y - 1.0
        a = a * y
    y = y - 1.0
    gamma_poly = 1.0 - 0.5748646*y + 0.9512363*y**2 - 0.6998588*y**3 + 0.4245549*y**4 - 0.1010678*y**5
    return a * gamma_poly

@njit(nopython=True)
def calculate_vhs_sigma_g(vr_mag):
    if vr_mag < 1e-9: return 0.0
    exponent = OMEGA_VHS - 0.5
    c_ref_sq = 2 * KB * T_REF_AR / MASS_AR
    gamma_val = gamma_function_approx(2.5 - OMEGA_VHS)
    sigma_g = (PI * D_REF_AR**2) * vr_mag * ((c_ref_sq / vr_mag**2)**exponent) / gamma_val
    return sigma_g

@njit(nopython=True)
def perform_post_collision(p1_idx, p2_idx, particles, rng_state):
    vr = particles[p1_idx, 1:4] - particles[p2_idx, 1:4]
    vr_mag = np.sqrt(np.sum(vr**2))
    if vr_mag < 1e-9: return False
    vcm = 0.5 * (particles[p1_idx, 1:4] + particles[p2_idx, 1:4])
    cos_chi = 2 * rng_state.random() - 1.0
    sin_chi = np.sqrt(1.0 - cos_chi**2)
    phi_chi = 2.0 * PI * rng_state.random()
    rand_vec = np.array([sin_chi * np.cos(phi_chi), sin_chi * np.sin(phi_chi), cos_chi])
    vr_prime = vr_mag * rand_vec
    particles[p1_idx, 1:4] = vcm + 0.5 * vr_prime
    particles[p2_idx, 1:4] = vcm - 0.5 * vr_prime
    return True

@njit(nopython=True)
def standard_sbt_scheme(particles, lx, indices_in_cell, cell_vol, dt, fnum, rng_state):
    n_accepted = 0
    num_particles_in_cell = len(indices_in_cell)
    if num_particles_in_cell < 2: return 0, 0, 0.0
    n_selected = float(num_particles_in_cell - 1)
    sum_sep = 0.0
    for i in range(num_particles_in_cell - 1):
        j_local = rng_state.integers(i + 1, num_particles_in_cell)
        p1_idx, p2_idx = indices_in_cell[i], indices_in_cell[j_local]
        vr = particles[p1_idx, 1:4] - particles[p2_idx, 1:4]
        vr_mag = np.sqrt(np.sum(vr**2))
        if vr_mag < 1e-9: continue
        sigma_g = calculate_vhs_sigma_g(vr_mag)
        multiplier = num_particles_in_cell - (i + 1)
        collision_prob = (multiplier * fnum * dt * sigma_g) / cell_vol
        if rng_state.random() < collision_prob:
            if perform_post_collision(p1_idx, p2_idx, particles, rng_state):
                n_accepted += 1
                delta_x = np.abs(particles[p1_idx, 0] - particles[p2_idx, 0])
                sum_sep += min(delta_x, lx - delta_x)
    return n_accepted, n_selected, sum_sep

@njit(nopython=True)
def perform_collisions(method, particles, lx, indices_in_cell, cell_vol, dt, fnum, rng_state, sigma_g_max_cell, n_sel_fraction, duplicate_check_array):
    n_accepted = 0
    n_selected = 0.0
    sum_separation = 0.0
    n_prob_exceed = 0  # Count collisions with probability > 1
    num_particles_in_cell = len(indices_in_cell)
    if num_particles_in_cell < 2: return 0, sigma_g_max_cell, 0.0, 0.0, 0

    if method == 1: # SBT
        prob_const = fnum * dt / cell_vol
        n_selected = num_particles_in_cell * (num_particles_in_cell - 1) / 2.0
        for i in range(num_particles_in_cell):
            for j in range(i + 1, num_particles_in_cell):
                p1_idx, p2_idx = indices_in_cell[i], indices_in_cell[j]
                vr = particles[p1_idx, 1:4] - particles[p2_idx, 1:4]
                vr_mag = np.sqrt(np.sum(vr**2))
                if vr_mag < 1e-9: continue
                sigma_g = calculate_vhs_sigma_g(vr_mag)
                collision_prob = prob_const * sigma_g
                if collision_prob > 1.0:
                    n_prob_exceed += 1
                if rng_state.random() < collision_prob:
                    if perform_post_collision(p1_idx, p2_idx, particles, rng_state):
                        n_accepted += 1
                        delta_x = np.abs(particles[p1_idx, 0] - particles[p2_idx, 0])
                        sum_separation += min(delta_x, lx - delta_x)
        return n_accepted, sigma_g_max_cell, n_selected, sum_separation, n_prob_exceed

    elif method == 2: # DCP
        num_pairs = num_particles_in_cell * (num_particles_in_cell - 1) // 2
        if num_pairs == 0: return 0, sigma_g_max_cell, 0.0, 0.0, 0
        pairs = np.empty((num_pairs, 2), dtype=np.int32)
        weights = np.empty(num_pairs, dtype=np.float64)
        pair_count = 0
        for i in range(num_particles_in_cell):
            for j in range(i + 1, num_particles_in_cell):
                p1_idx, p2_idx = indices_in_cell[i], indices_in_cell[j]
                delta_x = np.abs(particles[p1_idx, 0] - particles[p2_idx, 0])
                distance = min(delta_x, lx - delta_x)
                weights[pair_count] = 1.0 / (distance + 1e-10)
                pairs[pair_count, 0] = p1_idx
                pairs[pair_count, 1] = p2_idx
                pair_count += 1
        sum_of_sigma_g = 0.0
        for i in range(num_pairs):
            p1_idx, p2_idx = pairs[i,0], pairs[i,1]
            vr = particles[p1_idx, 1:4] - particles[p2_idx, 1:4]
            sum_of_sigma_g += calculate_vhs_sigma_g(np.sqrt(np.sum(vr**2)))
        expected_collisions = (sum_of_sigma_g * fnum * dt) / cell_vol
        num_collisions_to_perform = int(np.floor(expected_collisions + rng_state.random()))
        n_selected = float(num_collisions_to_perform)
        for _ in range(num_collisions_to_perform):
            if np.sum(weights) < 1e-12: break
            selected_idx = np.argmax(weights)
            p1_idx, p2_idx = pairs[selected_idx, 0], pairs[selected_idx, 1]
            if perform_post_collision(p1_idx, p2_idx, particles, rng_state):
                n_accepted += 1
                delta_x = np.abs(particles[p1_idx, 0] - particles[p2_idx, 0])
                sum_separation += min(delta_x, lx - delta_x)
            weights[selected_idx] = 0.0
        return n_accepted, sigma_g_max_cell, n_selected, sum_separation, n_prob_exceed

    elif method == 3: # NTC - Corrected with Pre-Scan to prevent P > 1
        # --- Pre-scan to find a better majorant for the current step ---
        num_prescan_samples = num_particles_in_cell
        for _ in range(num_prescan_samples):
            p1_idx_local = rng_state.integers(0, num_particles_in_cell)
            p2_idx_local = rng_state.integers(0, num_particles_in_cell)
            while p1_idx_local == p2_idx_local:
                p2_idx_local = rng_state.integers(0, num_particles_in_cell)

            p1_idx, p2_idx = indices_in_cell[p1_idx_local], indices_in_cell[p2_idx_local]
            vr = particles[p1_idx, 1:4] - particles[p2_idx, 1:4]
            sigma_g_current = calculate_vhs_sigma_g(np.sqrt(np.sum(vr**2)))

            if sigma_g_current > sigma_g_max_cell:
                sigma_g_max_cell = sigma_g_current
        # --- End of Pre-scan ---

        # Now calculate the number of pairs to select using the updated (more realistic) majorant
        num_pairs_float = 0.5 * num_particles_in_cell * (num_particles_in_cell - 1)
        if sigma_g_max_cell <= 0:
            sigma_g_max_cell = 1e-18  # Safety check
        num_pairs_to_select = int(num_pairs_float * fnum * sigma_g_max_cell * dt / cell_vol + rng_state.random())
        n_selected = float(num_pairs_to_select)

        # Main collision loop using the sophisticated NTC logic
        for _ in range(num_pairs_to_select):
            p1_idx_local = rng_state.integers(0, num_particles_in_cell)
            p2_idx_local = rng_state.integers(0, num_particles_in_cell)
            while p1_idx_local == p2_idx_local:
                p2_idx_local = rng_state.integers(0, num_particles_in_cell)

            p1_idx, p2_idx = indices_in_cell[p1_idx_local], indices_in_cell[p2_idx_local]
            vr = particles[p1_idx, 1:4] - particles[p2_idx, 1:4]
            sigma_g_current = calculate_vhs_sigma_g(np.sqrt(np.sum(vr**2)))

            # Since we pre-scanned, this condition should rarely be true after the initial steps,
            # but it's kept for robustness.
            if sigma_g_current > sigma_g_max_cell:
                sigma_g_max_cell = sigma_g_current
                if perform_post_collision(p1_idx, p2_idx, particles, rng_state):
                    n_accepted += 1
                    delta_x = np.abs(particles[p1_idx, 0] - particles[p2_idx, 0])
                    sum_separation += min(delta_x, lx - delta_x)
            # Standard probability check
            elif rng_state.random() < sigma_g_current / sigma_g_max_cell:
                if perform_post_collision(p1_idx, p2_idx, particles, rng_state):
                    n_accepted += 1
                    delta_x = np.abs(particles[p1_idx, 0] - particles[p2_idx, 0])
                    sum_separation += min(delta_x, lx - delta_x)
        return n_accepted, sigma_g_max_cell, n_selected, sum_separation, n_prob_exceed

    elif method == 4: # GBT
        N = num_particles_in_cell
        N_sel = int(n_sel_fraction)  # Now n_sel_fraction contains the actual N_sel value

        # Step 1: Check condition - if N_sel >= N^l - 1, use standard SBT scheme
        if N_sel >= N - 1 or N_sel < 1:
            n_acc, n_sel_fallback, sum_sep = standard_sbt_scheme(particles, lx, indices_in_cell, cell_vol, dt, fnum, rng_state)
            return n_acc, sigma_g_max_cell, n_sel_fallback, sum_sep, 0

        n_selected = float(N_sel)

        # Step 2: Algorithm A - Shuffling procedure to reorder particle list
        # Choose N_sel random particles and exchange positions with first N_sel particles
        all_local_indices = np.arange(N)
        for i in range(N_sel):
            k = rng_state.integers(i, N)  # Random selection from remaining particles
            all_local_indices[i], all_local_indices[k] = all_local_indices[k], all_local_indices[i]
        reordered_global_indices = indices_in_cell[all_local_indices]

        # GBT collision probability constants
        k_k_const = (N * (N - 1.0)) / (N_sel * (2.0 * N - N_sel - 1.0))
        prob_const = fnum * dt / cell_vol

        # Step 3: Main collision loop - process first N_sel particles in order
        for i in range(N_sel):
            if i + 1 >= N: continue
            j = rng_state.integers(i + 1, N)  # Select second particle from remaining
            p1_idx = reordered_global_indices[i]
            p2_idx = reordered_global_indices[j]
            vr = particles[p1_idx, 1:4] - particles[p2_idx, 1:4]
            vr_mag = np.sqrt(np.sum(vr**2))
            if vr_mag < 1e-9: continue
            sigma_g = calculate_vhs_sigma_g(vr_mag)
            k_k = k_k_const * (N - (i + 1.0))
            collision_prob = k_k * prob_const * sigma_g
            if collision_prob > 1.0:
                n_prob_exceed += 1
            if rng_state.random() < collision_prob:
                if perform_post_collision(p1_idx, p2_idx, particles, rng_state):
                    n_accepted += 1
                    delta_x = np.abs(particles[p1_idx, 0] - particles[p2_idx, 0])
                    sum_separation += min(delta_x, lx - delta_x)
        return n_accepted, sigma_g_max_cell, n_selected, sum_separation, n_prob_exceed

    elif method == 5: # SSBT
        prob_const = fnum * dt / cell_vol
        multiplier = (num_particles_in_cell - 1.0) / 2.0
        n_selected = float(num_particles_in_cell)
        for i_local in range(num_particles_in_cell):
            p1_idx = indices_in_cell[i_local]
            j_local = rng_state.integers(0, num_particles_in_cell)
            while j_local == i_local:
                j_local = rng_state.integers(0, num_particles_in_cell)
            p2_idx = indices_in_cell[j_local]
            vr = particles[p1_idx, 1:4] - particles[p2_idx, 1:4]
            vr_mag = np.sqrt(np.sum(vr**2))
            if vr_mag < 1e-9: continue
            sigma_g = calculate_vhs_sigma_g(vr_mag)
            collision_prob = multiplier * prob_const * sigma_g
            if collision_prob > 1.0:
                n_prob_exceed += 1
            if rng_state.random() < collision_prob:
                if perform_post_collision(p1_idx, p2_idx, particles, rng_state):
                    n_accepted += 1
                    delta_x = np.abs(particles[p1_idx, 0] - particles[p2_idx, 0])
                    sum_separation += min(delta_x, lx - delta_x)
        return n_accepted, sigma_g_max_cell, n_selected, sum_separation, n_prob_exceed

    elif method == 6: # SGBT
        N = num_particles_in_cell
        N_sel = int(n_sel_fraction)  # Now n_sel_fraction contains the actual N_sel value

        # Step 1: Check condition - if N_sel >= N^l - 1, use standard SBT scheme
        if N_sel >= N - 1 or N_sel < 1:
            n_acc, n_sel_fallback, sum_sep = standard_sbt_scheme(particles, lx, indices_in_cell, cell_vol, dt, fnum, rng_state)
            return n_acc, sigma_g_max_cell, n_sel_fallback, sum_sep, 0

        n_selected = float(N_sel)

        # Step 2: Shuffling procedure - reorder particle list putting N_sel selected particles first
        all_local_indices = np.arange(N)
        for i in range(N_sel):
            k = rng_state.integers(i, N)
            all_local_indices[i], all_local_indices[k] = all_local_indices[k], all_local_indices[i]
        reordered_global_indices = indices_in_cell[all_local_indices]

        prob_const = fnum * dt / cell_vol
        multiplier = (N * (N - 1.0)) / (N_sel * 2.0)

        # Step 3: Main collision loop - choose first N_sel particles in order
        for i in range(N_sel):
            p1_idx = reordered_global_indices[i]  # First particle from shuffled list

            # Step 4: Randomly select second particle j from all N particles (j ≠ i)
            j = rng_state.integers(0, N)
            while j == i:
                j = rng_state.integers(0, N)
            p2_idx = reordered_global_indices[j]

            # Step 5: Check for duplicate collision using both conditions as per algorithm
            # Check if I_p(i) = j AND I_p(j) = i (both conditions must be true for duplicate)
            if duplicate_check_array[p1_idx] == p2_idx and duplicate_check_array[p2_idx] == p1_idx:
                continue  # Reject as duplicated pair

            # Step 6: Calculate collision probability using VHS model
            vr = particles[p1_idx, 1:4] - particles[p2_idx, 1:4]
            vr_mag = np.sqrt(np.sum(vr**2))
            if vr_mag < 1e-9: continue
            sigma_g = calculate_vhs_sigma_g(vr_mag)
            collision_prob = multiplier * prob_const * sigma_g

            if collision_prob > 1.0:
                n_prob_exceed += 1

            # Step 7: Accept/reject collision and update velocities
            if rng_state.random() < collision_prob:
                if perform_post_collision(p1_idx, p2_idx, particles, rng_state):
                    n_accepted += 1
                    # Step 8: Mark pair in duplicate check array to prevent repeated collision
                    duplicate_check_array[p1_idx] = p2_idx
                    duplicate_check_array[p2_idx] = p1_idx
                    delta_x = np.abs(particles[p1_idx, 0] - particles[p2_idx, 0])
                    sum_separation += min(delta_x, lx - delta_x)
        return n_accepted, sigma_g_max_cell, n_selected, sum_separation, n_prob_exceed

    elif method == 7: # MFS - Corrected with Pre-Scan to prevent P > 1
        # --- Pre-scan to find a better majorant for the current step ---
        num_prescan_samples = num_particles_in_cell
        for _ in range(num_prescan_samples):
            p1_idx_local = rng_state.integers(0, num_particles_in_cell)
            p2_idx_local = rng_state.integers(0, num_particles_in_cell)
            while p1_idx_local == p2_idx_local:
                p2_idx_local = rng_state.integers(0, num_particles_in_cell)

            p1_idx, p2_idx = indices_in_cell[p1_idx_local], indices_in_cell[p2_idx_local]
            vr = particles[p1_idx, 1:4] - particles[p2_idx, 1:4]
            sigma_g_current = calculate_vhs_sigma_g(np.sqrt(np.sum(vr**2)))

            if sigma_g_current > sigma_g_max_cell:
                sigma_g_max_cell = sigma_g_current
        # --- End of Pre-scan ---

        # Main MFS collision loop with time stepping
        cell_time = 0.0
        n_sel_counter = 0
        while cell_time < dt:
            num_pairs_float = 0.5 * num_particles_in_cell * (num_particles_in_cell - 1)
            if num_pairs_float < 1 or sigma_g_max_cell <= 0:
                break
            nu_max = num_pairs_float * fnum * sigma_g_max_cell / cell_vol
            delta_t_c = -np.log(1.0 - rng_state.random()) / nu_max
            cell_time += delta_t_c
            n_sel_counter += 1
            if cell_time < dt:
                p1_idx_local = rng_state.integers(0, num_particles_in_cell)
                p2_idx_local = rng_state.integers(0, num_particles_in_cell)
                while p1_idx_local == p2_idx_local:
                    p2_idx_local = rng_state.integers(0, num_particles_in_cell)
                p1_idx, p2_idx = indices_in_cell[p1_idx_local], indices_in_cell[p2_idx_local]
                vr = particles[p1_idx, 1:4] - particles[p2_idx, 1:4]
                sigma_g_current = calculate_vhs_sigma_g(np.sqrt(np.sum(vr**2)))

                # Since we pre-scanned, this condition should rarely be true after initial steps
                if sigma_g_current > sigma_g_max_cell:
                    sigma_g_max_cell = sigma_g_current
                    if perform_post_collision(p1_idx, p2_idx, particles, rng_state):
                        n_accepted += 1
                        delta_x = np.abs(particles[p1_idx, 0] - particles[p2_idx, 0])
                        sum_separation += min(delta_x, lx - delta_x)
                elif rng_state.random() < sigma_g_current / sigma_g_max_cell:
                    if perform_post_collision(p1_idx, p2_idx, particles, rng_state):
                        n_accepted += 1
                        delta_x = np.abs(particles[p1_idx, 0] - particles[p2_idx, 0])
                        sum_separation += min(delta_x, lx - delta_x)
        return n_accepted, sigma_g_max_cell, float(n_sel_counter), sum_separation, n_prob_exceed

    elif method == 8: # NN - Corrected with Pre-Scan to prevent P > 1
        # --- Pre-scan to find a better majorant for the current step ---
        num_prescan_samples = num_particles_in_cell
        for _ in range(num_prescan_samples):
            p1_idx_local = rng_state.integers(0, num_particles_in_cell)
            p2_idx_local = rng_state.integers(0, num_particles_in_cell)
            while p1_idx_local == p2_idx_local:
                p2_idx_local = rng_state.integers(0, num_particles_in_cell)

            p1_idx, p2_idx = indices_in_cell[p1_idx_local], indices_in_cell[p2_idx_local]
            vr = particles[p1_idx, 1:4] - particles[p2_idx, 1:4]
            sigma_g_current = calculate_vhs_sigma_g(np.sqrt(np.sum(vr**2)))

            if sigma_g_current > sigma_g_max_cell:
                sigma_g_max_cell = sigma_g_current
        # --- End of Pre-scan ---

        # Now calculate pairs to select using the updated (more realistic) majorant
        num_pairs_float = 0.5 * num_particles_in_cell * (num_particles_in_cell - 1)
        if sigma_g_max_cell <= 0:
            sigma_g_max_cell = 1e-18  # Safety check
        num_pairs_to_select = int(num_pairs_float * fnum * sigma_g_max_cell * dt / cell_vol + rng_state.random())
        n_selected = float(num_pairs_to_select)
        if n_selected == 0:
            return 0, sigma_g_max_cell, 0.0, 0.0, 0

        # Build nearest neighbor map
        N = num_particles_in_cell
        cell_positions = particles[indices_in_cell, 0]
        dist_matrix = np.full((N, N), np.finfo(np.float64).max, dtype=np.float64)
        for i in range(N):
            for j in range(i + 1, N):
                delta_x = np.abs(cell_positions[i] - cell_positions[j])
                dist = min(delta_x, lx - delta_x)
                dist_matrix[i, j] = dist
                dist_matrix[j, i] = dist
        nearest_neighbor_map = np.argmin(dist_matrix, axis=1)

        # Main NN collision loop using nearest neighbors
        for _ in range(num_pairs_to_select):
            p1_local_idx = rng_state.integers(0, N)
            p2_local_idx = nearest_neighbor_map[p1_local_idx]
            p1_idx, p2_idx = indices_in_cell[p1_local_idx], indices_in_cell[p2_local_idx]
            vr = particles[p1_idx, 1:4] - particles[p2_idx, 1:4]
            sigma_g_current = calculate_vhs_sigma_g(np.sqrt(np.sum(vr**2)))

            # Since we pre-scanned, this condition should rarely be true after initial steps
            if sigma_g_current > sigma_g_max_cell:
                sigma_g_max_cell = sigma_g_current
                if perform_post_collision(p1_idx, p2_idx, particles, rng_state):
                    n_accepted += 1
                    sum_separation += dist_matrix[p1_local_idx, p2_local_idx]
            elif rng_state.random() < sigma_g_current / sigma_g_max_cell:
                if perform_post_collision(p1_idx, p2_idx, particles, rng_state):
                    n_accepted += 1
                    sum_separation += dist_matrix[p1_local_idx, p2_local_idx]
        return n_accepted, sigma_g_max_cell, n_selected, sum_separation, n_prob_exceed

    return 0, sigma_g_max_cell, 0.0, 0.0, 0

@njit(nopython=True)
def calculate_theoretical_collision_frequency(avg_temp, n_density):
    d_ref_sq = D_REF_AR**2
    temp_ratio_term = (avg_temp / T_REF_AR)**(1.0 - OMEGA_VHS)
    sqrt_term = np.sqrt(PI * KB * T_REF_AR / MASS_AR)
    cf_th = 4 * n_density * d_ref_sq * sqrt_term * temp_ratio_term
    return cf_th

def calculate_numerical_collision_frequency(num_sim_collisions, num_particles, time_interval):
    if num_particles == 0 or time_interval == 0:
        return 0.0
    cf_num = num_sim_collisions / (0.5 * num_particles * time_interval)
    return cf_num

def calculate_simulation_parameters(particles_per_cell, n_sel_for_gbt_sgbt=None):
    """
    Calculate and validate simulation parameters based on particles per cell and N_sel.
    Adjusts both cell size and time step to satisfy all constraints.
    """
    global PARTICLES_PER_CELL_INIT, TOTAL_PARTICLES_SIM, FNUM, DT, NUM_CELLS_X, CELL_VOLUME_CONCEPTUAL, CENTRAL_CELL_IDX

    # Update global parameters
    PARTICLES_PER_CELL_INIT = particles_per_cell

    # Physical constants
    sigma_ref = PI * D_REF_AR**2
    mfp = 1 / (np.sqrt(2) * sigma_ref * N_DENSITY_REAL)  # Mean free path
    cm = np.sqrt(2 * KB * T_INIT / MASS_AR)  # Most probable thermal velocity
    tc = mfp / cm  # Mean collision time

    print(f"\n=== Physical Parameters ===")
    print(f"Mean free path (λ): {mfp*1e6:.3f} μm")
    print(f"Most probable velocity (cm): {cm:.1f} m/s")
    print(f"Mean collision time (tc): {tc*1e12:.3f} ps")

    # Show specific multiplier calculations for GBT/SGBT if relevant
    if n_sel_for_gbt_sgbt is not None:
        N = particles_per_cell
        N_sel = n_sel_for_gbt_sgbt
        if N_sel < N - 1 and N_sel >= 1:
            gbt_mult_const = (N * (N - 1.0)) / (N_sel * (2.0 * N - N_sel - 1.0))
            gbt_mult_max = gbt_mult_const * (N - 1.0)
            sgbt_mult = (N * (N - 1.0)) / (N_sel * 2.0)
            print(f"For N={N}, N_sel={N_sel}:")
            print(f"  GBT multiplier: constant={gbt_mult_const:.2f}, maximum={gbt_mult_max:.2f}")
            print(f"  SGBT multiplier: {sgbt_mult:.2f}")
            print(f"  This means GBT collision probability = {gbt_mult_max:.2f} × base_probability")

    # Start with reasonable initial values
    NUM_CELLS_X = 100
    max_iterations = 50
    safety_factor = 0.90  # Keep probabilities below 90% to be safer

    # Estimate maximum sigma_g (worst case scenario)
    max_sigma_g_estimate = PI * D_REF_AR**2 * np.sqrt(8 * KB * T_INIT / (PI * MASS_AR))

    # Pre-calculate GBT/SGBT multipliers if needed for more aggressive initial scaling
    initial_scaling_factor = 1.0
    if n_sel_for_gbt_sgbt is not None:
        N = particles_per_cell
        N_sel = n_sel_for_gbt_sgbt
        if N_sel < N - 1 and N_sel >= 1:
            gbt_mult_const = (N * (N - 1.0)) / (N_sel * (2.0 * N - N_sel - 1.0))
            gbt_mult_max = gbt_mult_const * (N - 1.0)
            sgbt_mult = (N * (N - 1.0)) / (N_sel * 2.0)
            max_mult = max(gbt_mult_max, sgbt_mult)
            # Start with more cells if we expect high multipliers
            if max_mult > 5.0:
                initial_scaling_factor = np.sqrt(max_mult / 2.0)
                NUM_CELLS_X = int(NUM_CELLS_X * initial_scaling_factor)
                print(f"Pre-scaling cells by {initial_scaling_factor:.2f} due to high multiplier {max_mult:.2f}")

    print(f"\n=== Iterative Parameter Adjustment ===")

    for iteration in range(max_iterations):
        # Calculate current cell parameters
        cell_width = LX / NUM_CELLS_X
        CELL_VOLUME_CONCEPTUAL = cell_width  # 1D volume is just length
        TOTAL_PARTICLES_SIM = int(NUM_CELLS_X * PARTICLES_PER_CELL_INIT)
        FNUM = (N_DENSITY_REAL * CELL_VOLUME_CONCEPTUAL) / PARTICLES_PER_CELL_INIT

        # Calculate time step to maintain dx/dt = 1
        # dx/dt = 1 means: (cell_width/λ) / (dt/tc) = 1
        # Therefore: dt = tc * (cell_width/λ)
        dx_over_mfp = cell_width / mfp
        DT = tc * dx_over_mfp
        dt_over_tc = DT / tc

        # Check physical constraints
        constraint1 = dx_over_mfp < (1.0/3.0)  # cell_width < λ/3
        constraint2 = dt_over_tc < 1.0         # dt < tc
        constraint3 = abs((dx_over_mfp / dt_over_tc) - 1.0) < 1e-6  # dx/dt = 1

        # Calculate collision probabilities
        base_prob = (FNUM * DT * max_sigma_g_estimate) / CELL_VOLUME_CONCEPTUAL

        # SBT probability
        sbt_prob = base_prob

        # GBT probability (worst case with minimum N_sel)
        if n_sel_for_gbt_sgbt is not None:
            N = PARTICLES_PER_CELL_INIT
            N_sel = n_sel_for_gbt_sgbt
            if N_sel < N - 1 and N_sel >= 1:
                # GBT multiplier - MAXIMUM possible value (when i=0)
                gbt_multiplier_const = (N * (N - 1.0)) / (N_sel * (2.0 * N - N_sel - 1.0))
                gbt_multiplier_max = gbt_multiplier_const * (N - 1.0)  # Maximum occurs at i=0
                gbt_prob = gbt_multiplier_max * base_prob

                # SGBT multiplier
                sgbt_multiplier = (N * (N - 1.0)) / (N_sel * 2.0)
                sgbt_prob = sgbt_multiplier * base_prob
            else:
                # Will fall back to SBT
                gbt_prob = sbt_prob
                sgbt_prob = sbt_prob
        else:
            gbt_prob = sbt_prob
            sgbt_prob = sbt_prob

        # Find maximum probability
        max_prob = max(sbt_prob, gbt_prob, sgbt_prob)

        # Check if all constraints are satisfied
        all_constraints_ok = constraint1 and constraint2 and constraint3
        prob_constraint_ok = max_prob < safety_factor

        if iteration == 0:
            print(f"Initial attempt:")
        else:
            print(f"Iteration {iteration}:")

        print(f"  Cells: {NUM_CELLS_X}, Cell width: {cell_width*1e6:.3f} μm, dt: {DT*1e12:.3f} ps")
        print(f"  dx/λ = {dx_over_mfp:.4f} (<1/3? {constraint1}), dt/tc = {dt_over_tc:.4f} (<1? {constraint2})")
        print(f"  dx/dt = {dx_over_mfp/dt_over_tc:.6f} (=1? {constraint3})")
        print(f"  FNUM = {FNUM:.6e}, base_prob = {base_prob:.6f}")
        print(f"  Max collision prob: {max_prob:.4f} (<{safety_factor}? {prob_constraint_ok})")

        if n_sel_for_gbt_sgbt is not None:
            print(f"  SBT: {sbt_prob:.4f}, GBT: {gbt_prob:.4f}, SGBT: {sgbt_prob:.4f}")
            if N_sel < PARTICLES_PER_CELL_INIT - 1 and N_sel >= 1:
                N = PARTICLES_PER_CELL_INIT
                gbt_mult_const = (N * (N - 1.0)) / (N_sel * (2.0 * N - N_sel - 1.0))
                gbt_mult_max = gbt_mult_const * (N - 1.0)
                sgbt_mult = (N * (N - 1.0)) / (N_sel * 2.0)
                print(f"    GBT: const={gbt_mult_const:.3f}, max_mult={gbt_mult_max:.3f}")
                print(f"    SGBT: mult={sgbt_mult:.3f}")
                print(f"    Calculation: {gbt_mult_max:.3f} × {base_prob:.6f} = {gbt_prob:.4f}")

        if all_constraints_ok and prob_constraint_ok:
            print(f"✓ All constraints satisfied after {iteration+1} iterations!")
            break

        # If constraints not satisfied, adjust NUM_CELLS_X
        # Key insight: collision_prob scales as 1/NUM_CELLS_X^2 because:
        # - FNUM ∝ 1/NUM_CELLS_X (smaller cells, lower FNUM)
        # - DT ∝ 1/NUM_CELLS_X (to maintain dx/dt=1)
        # - CELL_VOLUME ∝ 1/NUM_CELLS_X
        # So prob = (FNUM*DT)/CELL_VOLUME ∝ (1/NUM_CELLS_X^2)/(1/NUM_CELLS_X) = 1/NUM_CELLS_X^2

        if not constraint1 or not prob_constraint_ok:
            old_num_cells = NUM_CELLS_X

            if not constraint1:
                # Ensure dx/λ < 1/3
                required_cells_for_dx = int(np.ceil(NUM_CELLS_X * dx_over_mfp / (1.0/3.0 * 0.90)))
                NUM_CELLS_X = max(NUM_CELLS_X + 20, required_cells_for_dx)
                print(f"    Adjusting for dx/λ constraint: {old_num_cells} → {NUM_CELLS_X}")

            if not prob_constraint_ok:
                # To reduce collision probability by factor prob_reduction_factor,
                # increase NUM_CELLS_X by sqrt(prob_reduction_factor) due to 1/NUM_CELLS_X^2 scaling
                prob_reduction_factor = max_prob / safety_factor
                cells_multiplier = np.sqrt(prob_reduction_factor)

                # For very high multipliers, be even more aggressive
                if max_prob > 5.0:
                    cells_multiplier *= 1.5  # Extra safety margin
                elif max_prob > 2.0:
                    cells_multiplier *= 1.2

                new_cells = max(NUM_CELLS_X + 50, int(NUM_CELLS_X * cells_multiplier))
                print(f"    Adjusting for probability constraint: {NUM_CELLS_X} → {new_cells}")
                print(f"    Probability reduction needed: {prob_reduction_factor:.3f}, cells multiplier: {cells_multiplier:.3f}")
                NUM_CELLS_X = new_cells
        elif not constraint2:
            # dt/tc constraint will be automatically satisfied when dx/dt=1 and dx/λ<1/3
            NUM_CELLS_X = int(NUM_CELLS_X * 1.3)
            print(f"    Adjusting for dt/tc constraint: {NUM_CELLS_X}")

        # Safety check to prevent infinite loops
        if iteration > 10 and NUM_CELLS_X > 10000:
            print(f"⚠ Warning: Very high number of cells ({NUM_CELLS_X}) - may indicate fundamental issue")
            break

        if iteration == max_iterations - 1:
            print(f"⚠ Warning: Could not satisfy all constraints after {max_iterations} iterations")
            print(f"⚠ Proceeding with current parameters - expect probability exceed issues")

    # Update central cell index (will be set properly in run_dsmc_simulation)
    CENTRAL_CELL_IDX = NUM_CELLS_X // 2

    # Final parameter calculations
    cell_width = LX / NUM_CELLS_X
    CELL_VOLUME_CONCEPTUAL = cell_width
    TOTAL_PARTICLES_SIM = int(NUM_CELLS_X * PARTICLES_PER_CELL_INIT)
    FNUM = (N_DENSITY_REAL * CELL_VOLUME_CONCEPTUAL) / PARTICLES_PER_CELL_INIT

    dx_over_mfp = cell_width / mfp
    DT = tc * dx_over_mfp
    dt_over_tc = DT / tc
    dx_dt_ratio = dx_over_mfp / dt_over_tc

    # Final collision probability calculations with same logic as iteration
    base_prob = (FNUM * DT * max_sigma_g_estimate) / CELL_VOLUME_CONCEPTUAL
    final_sbt_prob = base_prob

    if n_sel_for_gbt_sgbt is not None:
        N = PARTICLES_PER_CELL_INIT
        N_sel = n_sel_for_gbt_sgbt
        if N_sel < N - 1 and N_sel >= 1:
            # GBT multiplier - MAXIMUM possible value
            gbt_multiplier_const = (N * (N - 1.0)) / (N_sel * (2.0 * N - N_sel - 1.0))
            gbt_multiplier_max = gbt_multiplier_const * (N - 1.0)  # Maximum occurs at i=0
            final_gbt_prob = gbt_multiplier_max * base_prob

            # SGBT multiplier
            sgbt_multiplier = (N * (N - 1.0)) / (N_sel * 2.0)
            final_sgbt_prob = sgbt_multiplier * base_prob
        else:
            final_gbt_prob = final_sbt_prob
            final_sgbt_prob = final_sbt_prob
    else:
        final_gbt_prob = final_sbt_prob
        final_sgbt_prob = final_sbt_prob

    # Report final parameters
    print(f"\n=== Final Simulation Parameters ===")
    print(f"Particles per cell: {PARTICLES_PER_CELL_INIT}")
    if n_sel_for_gbt_sgbt is not None:
        print(f"N_sel for GBT/SGBT: {n_sel_for_gbt_sgbt}")
    print(f"Total particles: {TOTAL_PARTICLES_SIM}")
    print(f"Number of cells: {NUM_CELLS_X}")
    print(f"Cell width: {cell_width*1e6:.3f} μm")
    print(f"Time step (dt): {DT*1e12:.3f} ps")

    print(f"\nPhysical Constraints:")
    print(f"Cell size / λ ratio: {dx_over_mfp:.4f} (< 1/3 = {dx_over_mfp < 1.0/3.0})")
    print(f"dt / tc ratio: {dt_over_tc:.4f} (< 1 = {dt_over_tc < 1.0})")
    print(f"Normalized dx/dt: {dx_dt_ratio:.6f} (≈ 1)")

    print(f"\nCollision Probabilities (estimated max):")
    print(f"  SBT/SSBT: {final_sbt_prob:.4f}")
    if n_sel_for_gbt_sgbt is not None:
        print(f"  GBT: {final_gbt_prob:.4f}")
        print(f"  SGBT: {final_sgbt_prob:.4f}")

        # Show the multiplier details for transparency
        if n_sel_for_gbt_sgbt < PARTICLES_PER_CELL_INIT - 1 and n_sel_for_gbt_sgbt >= 1:
            N = PARTICLES_PER_CELL_INIT
            N_sel = n_sel_for_gbt_sgbt
            gbt_mult_const = (N * (N - 1.0)) / (N_sel * (2.0 * N - N_sel - 1.0))
            gbt_mult_max = gbt_mult_const * (N - 1.0)
            sgbt_mult = (N * (N - 1.0)) / (N_sel * 2.0)
            print(f"  Detailed calculation for N={N}, N_sel={N_sel}:")
            print(f"    Base probability: {base_prob:.8f}")
            print(f"    GBT multiplier: const={gbt_mult_const:.4f}, max={gbt_mult_max:.4f}")
            print(f"    SGBT multiplier: {sgbt_mult:.4f}")
            print(f"    Final GBT prob: {gbt_mult_max:.4f} × {base_prob:.8f} = {final_gbt_prob:.6f}")
            print(f"    Final SGBT prob: {sgbt_mult:.4f} × {base_prob:.8f} = {final_sgbt_prob:.6f}")

            # Show the probability reduction achieved
            if initial_scaling_factor > 1.0:
                original_base_prob = base_prob * (NUM_CELLS_X / 100)**2  # Reverse the scaling
                original_gbt_prob = gbt_mult_max * original_base_prob
                print(f"    Without scaling: GBT prob would be {original_gbt_prob:.4f}")
                print(f"    Scaling factor applied: {NUM_CELLS_X/100:.2f}x cells, {original_gbt_prob/final_gbt_prob:.2f}x prob reduction")

    max_final_prob = max(final_sbt_prob, final_gbt_prob, final_sgbt_prob)
    if max_final_prob < 1.0:
        print(f"✓ All collision probabilities < 1.0")
    else:
        print(f"⚠ Maximum probability {max_final_prob:.4f} > 1.0 - expect issues!")

    print(f"=====================================\n")

    return DT

def initialize_particles(rng_state):
    particles = np.zeros((TOTAL_PARTICLES_SIM, 4))
    print("Initializing particles with Maxwell-Boltzmann velocity distribution...")
    particles[:, 0] = rng_state.random(TOTAL_PARTICLES_SIM) * LX
    v_thermal_std = np.sqrt(KB * T_INIT / MASS_AR)
    particles[:, 1:4] = rng_state.normal(0, v_thermal_std, (TOTAL_PARTICLES_SIM, 3))
    particles[:, 1:4] -= np.mean(particles[:, 1:4], axis=0)
    print(f"Initialized {TOTAL_PARTICLES_SIM} particles.")
    return particles

def plot_results(sampled_speeds, final_temp, method_name, freq_ratio_history, time_history, sof_history, acceptance_ratio, mfp, prob_exceed_history):
    plt.rcParams.update({'font.size': 16, 'axes.titlesize': 18, 'axes.labelsize': 16,
                         'xtick.labelsize': 14, 'ytick.labelsize': 14, 'legend.fontsize': 14})
    fig, axes = plt.subplots(2, 2, figsize=(22, 18))
    fig.suptitle(f'DSMC Relaxation Analysis (Method: {method_name})', fontsize=22)
    ax1, ax2, ax3, ax4 = axes.flatten()

    # Plot 1: Final Speed Distribution
    if len(sampled_speeds) > 0:
        ax1.hist(sampled_speeds, bins=100, density=True, label='DSMC Results', alpha=0.7, color='dodgerblue')
        v_max_range = np.max(sampled_speeds) * 1.15
        v_theory = np.linspace(0, v_max_range, 500)
        pv_theory = (4 * np.pi * (MASS_AR / (2 * np.pi * KB * final_temp))**1.5 * v_theory**2 * np.exp(-MASS_AR * v_theory**2 / (2 * KB * final_temp)))
        ax1.plot(v_theory, pv_theory, 'r-', linewidth=2.5, label=f'Maxwell-Boltzmann (T={final_temp:.1f}K)')
    ax1.set_xlabel('Speed (m/s)'); ax1.set_ylabel('Probability Density'); ax1.set_title('Final Speed Distribution')
    ax1.legend(); ax1.grid(True, linestyle=':')

    # Plot 2: Collision Frequency Ratio vs. Time
    if time_history and freq_ratio_history:
        plot_len = min(len(time_history), len(freq_ratio_history))
        ax2.plot(np.array(time_history[:plot_len]) * 1e9, freq_ratio_history[:plot_len], 'g-')
        ax2.axhline(1.0, color='r', linestyle='--', label='Ideal Ratio = 1.0')
    ax2.set_xlabel('Time (ns)'); ax2.set_ylabel('Collision Frequency Ratio'); ax2.set_title('Evolution of Collision Frequency Ratio')
    ax2.legend(); ax2.grid(True, linestyle=':'); ax2.set_ylim(bottom=0)

    # Plot 3: Mean Separation Distance vs. Time
    if time_history and sof_history:
        plot_len = min(len(time_history), len(sof_history))
        ax3.plot(np.array(time_history[:plot_len]) * 1e9, sof_history[:plot_len], 'm-')
    ax3.set_xlabel('Time (ns)'); ax3.set_ylabel('Mean Separation / MFP'); ax3.set_title('Evolution of Collision Separation (SoF)')
    ax3.grid(True, linestyle=':'); ax3.set_ylim(bottom=0)

    # Plot 4: Probability Exceed Ratio vs. Time
    if time_history and prob_exceed_history:
        plot_len = min(len(time_history), len(prob_exceed_history))
        ax4.plot(np.array(time_history[:plot_len]) * 1e9, [ratio * 100 for ratio in prob_exceed_history[:plot_len]], 'orange', linewidth=2)
    ax4.set_xlabel('Time (ns)'); ax4.set_ylabel('Probability Exceed Ratio (%)'); ax4.set_title('Percentage of Collisions with P > 1')
    ax4.grid(True, linestyle=':'); ax4.set_ylim(bottom=0)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    output_filename = f'DSMC_Combined_Results_{method_name}.eps'
    plt.savefig(output_filename, format='eps')
    print(f"\nCombined results plot saved as {output_filename}")
    plt.show()

def plot_advanced_stats(density_fluctuation_hist, temporal_corr_sums_multi, spatial_corr_sums, nsmpt_multi, nsmpx, navg_theory, method_name, dt, cell_width, cells_to_analyze):
    plt.rcParams.update({'font.size': 16, 'axes.titlesize': 18, 'axes.labelsize': 16, 'xtick.labelsize': 14, 'ytick.labelsize': 14, 'legend.fontsize': 14})
    fig, axes = plt.subplots(3, 1, figsize=(15, 24))
    fig.suptitle(f'Advanced Statistical Analysis (Method: {method_name})', fontsize=20)

    # Plot 1: Density Fluctuation Distribution
    ax1 = axes[0]
    dev_range = np.arange(-MAXD, MAXD + 1)
    total_samples = np.sum(density_fluctuation_hist)
    if total_samples > 0:
        prob_density_fluc = density_fluctuation_hist / total_samples
        ax1.bar(dev_range, prob_density_fluc, width=1.0, label='DSMC Simulation')
        k_values = dev_range + navg_theory
        valid_indices = k_values >= 0
        valid_k = k_values[valid_indices].astype(np.int64)
        if navg_theory > 0:
             log_poisson = valid_k * np.log(navg_theory) - navg_theory - gammaln(valid_k + 1)
             poisson_prob = np.zeros_like(k_values, dtype=float)
             poisson_prob[valid_indices] = np.exp(log_poisson)
             ax1.plot(dev_range, poisson_prob, 'r--', label='Poisson Theory')
    ax1.set_title('Number Density Fluctuation Distribution'); ax1.set_xlabel('Deviation from Mean (N - <N>)'); ax1.set_ylabel('Probability')
    ax1.legend(); ax1.grid(True, linestyle=':'); ax1.set_xlim(left=-MAXD, right=MAXD)

    # Plot 2: Multi-Cell Temporal Auto-Correlation (Enhanced)
    ax2 = axes[1]
    time_lags = np.arange(-MTC, MTC + 1) * dt * SAMPLING_INTERVAL
    prop_labels = ['N', 'u', 'v', 'w', 'T']
    colors = ['b', 'g', 'r', 'c', 'm']
    cell_colors = ['red', 'blue', 'green', 'orange', 'purple']

    if nsmpt_multi > MTC:  # Only plot if we have enough data
        print(f"Temporal correlation samples used: {nsmpt_multi - MTC}")

        # Apply smoothing filter for better visualization
        def smooth_data(data, window_size=3):
            """Simple moving average smoothing"""
            if len(data) < window_size:
                return data
            smoothed = np.zeros_like(data)
            for i in range(len(data)):
                start_idx = max(0, i - window_size//2)
                end_idx = min(len(data), i + window_size//2 + 1)
                smoothed[i] = np.mean(data[start_idx:end_idx])
            return smoothed

        # Plot each property for all cells with smoothing
        for prop_idx in range(5):
            for cell_idx, cell_num in enumerate(cells_to_analyze):
                # Proper normalization: R(τ) = C(τ) / C(0)
                zero_lag_variance = temporal_corr_sums_multi[cell_idx, prop_idx, MTC] / (nsmpt_multi - MTC)
                if abs(zero_lag_variance) > 1e-9:
                    normalized_corr = temporal_corr_sums_multi[cell_idx, prop_idx, :] / (nsmpt_multi - MTC) / zero_lag_variance

                    # Apply smoothing
                    smoothed_corr = smooth_data(normalized_corr, window_size=5)

                    # Different line styles for different cells
                    line_style = '-' if cell_idx == 0 else '--' if cell_idx == 1 else '-.' if cell_idx == 2 else ':' if cell_idx == 3 else '-'
                    alpha = 0.9 if cell_idx == 0 else 0.7  # Make first cell more prominent
                    linewidth = 2.0 if cell_idx == 0 else 1.5

                    # Only show labels for first cell to avoid clutter
                    label = f'{prop_labels[prop_idx]}' if cell_idx == 0 else ""

                    ax2.plot(time_lags * 1e9, smoothed_corr,
                            color=colors[prop_idx], linestyle=line_style, alpha=alpha,
                            label=label, linewidth=linewidth,
                            marker='o' if cell_idx == 0 and prop_idx < 2 else None, markersize=3)
    else:
        ax2.text(0.5, 0.5, f'Insufficient data: {nsmpt_multi} samples (need > {MTC})',
                transform=ax2.transAxes, ha='center', va='center', fontsize=14,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.8))

    ax2.set_title(f'Multi-Cell Temporal Auto-Correlation (Smoothed, N_samples={nsmpt_multi-MTC if nsmpt_multi > MTC else 0})');
    ax2.set_xlabel('Time Lag, τ (ns)'); ax2.set_ylabel('Normalized Correlation, R(τ)')
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); ax2.grid(True, linestyle=':')
    ax2.axhline(y=0, color='k', linestyle='-', alpha=0.3)
    ax2.axvline(x=0, color='k', linestyle='-', alpha=0.3)
    ax2.set_ylim(-1.1, 1.1)  # Fix y-axis limits for better comparison

    # Add text annotation explaining what we see
    info_text = f"Cells: {cells_to_analyze}\nSamples: {nsmpt_multi-MTC if nsmpt_multi > MTC else 0}\nSmoothing: 5-point moving avg"
    ax2.text(0.02, 0.02, info_text, transform=ax2.transAxes,
             verticalalignment='bottom', fontsize=9,
             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

    # Plot 3: Spatial Correlation (unchanged)
    ax3 = axes[2]
    space_lags = np.arange(-MXC, MXC + 1) * cell_width
    if nsmpx > 0:
        for i in range(5):
            zero_lag_variance = spatial_corr_sums[i, MXC] / nsmpx
            if abs(zero_lag_variance) > 1e-9:
                normalized_corr = spatial_corr_sums[i, :] / nsmpx / zero_lag_variance
                ax3.plot(space_lags * 1e6, normalized_corr, 'o-', color=colors[i], label=prop_labels[i], markersize=3)
    ax3.set_title(f'Spatial Correlation Relative to Cell {CENTRAL_CELL_IDX}'); ax3.set_xlabel('Spatial Lag, Δx (μm)'); ax3.set_ylabel('Normalized Correlation, G(Δx)')
    ax3.legend(); ax3.grid(True, linestyle=':')
    ax3.axhline(y=0, color='k', linestyle='-', alpha=0.3)
    ax3.axvline(x=0, color='k', linestyle='-', alpha=0.3)

    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    output_filename = f'DSMC_Advanced_Stats_{method_name}.eps'
    plt.savefig(output_filename, format='eps', bbox_inches='tight')
    print(f"Advanced stats plot saved as {output_filename}")
    plt.show()

def run_dsmc_simulation(method_code, n_sel_fraction):
    method_map = {1: 'SBT', 2: 'DCP', 3: 'NTC', 4: 'GBT', 5: 'SSBT', 6: 'SGBT', 7: 'MFS', 8: 'NN'}
    collision_method_name = method_map.get(method_code, 'Unknown')
    print(f"--- DSMC Relaxation using method: {collision_method_name} ---")

    rng_state = np.random.default_rng(seed=42)
    particles = initialize_particles(rng_state)
    sigma_g_max = np.full(NUM_CELLS_X, 1e-18, dtype=np.float64)

    total_accepted_collisions, total_selected_pairs, total_collision_separation = 0.0, 0.0, 0.0
    total_prob_exceed_collisions = 0.0
    last_sampled_total_collisions = 0.0
    last_sampled_prob_exceed = 0.0

    density_fluctuation_hist = np.zeros(2 * MAXD + 1, dtype=np.int64)
    navg_theory = PARTICLES_PER_CELL_INIT

    # Select multiple cells for temporal correlation analysis
    NUM_CELLS_TO_ANALYZE = 5  # Number of cells to analyze
    CENTRAL_CELL_IDX = NUM_CELLS_X // 2  # Update central cell index

    # Choose cells distributed across the domain
    if NUM_CELLS_X >= 5:
        cells_to_analyze = [
            NUM_CELLS_X // 4,           # Quarter cell
            CENTRAL_CELL_IDX,           # Center cell
            3 * NUM_CELLS_X // 4,       # Three-quarter cell
            NUM_CELLS_X // 8,           # One-eighth cell
            7 * NUM_CELLS_X // 8        # Seven-eighth cell
        ]
    else:
        # For small domains, just use available cells
        cells_to_analyze = list(range(min(NUM_CELLS_TO_ANALYZE, NUM_CELLS_X)))
        CENTRAL_CELL_IDX = cells_to_analyze[len(cells_to_analyze)//2] if cells_to_analyze else 0

    print(f"Analyzing temporal correlation for cells: {cells_to_analyze}")
    print(f"Central cell for spatial correlation: {CENTRAL_CELL_IDX}")

    # Multi-cell temporal correlation storage
    temporal_samples_storage_multi = np.zeros((len(cells_to_analyze), 5, 2 * MTC + 1))
    temporal_corr_sums_multi = np.zeros((len(cells_to_analyze), 5, 2 * MTC + 1))
    nsmpt_multi = 0

    spatial_corr_sums = np.zeros((5, 2 * MXC + 1))
    nsmpx = 0

    # Memory-efficient data collection with limits
    MAX_HISTORY_SAMPLES = 2000  # Limit history to reasonable size
    MAX_SPEED_SAMPLES = 50000   # Limit speed samples

    time_history, freq_ratio_history, sof_history, prob_exceed_history = [], [], [], []
    sampled_speeds_accumulator = []
    history_sample_count = 0

    start_time = time.time()
    num_steps = int(TOTAL_TIME / DT)
    speed_sampling_start_step = int(TOTAL_TIME * 0.9 / DT)
    cell_width = LX / NUM_CELLS_X
    sigma_ref = PI * D_REF_AR**2
    mfp = 1 / (np.sqrt(2) * sigma_ref * N_DENSITY_REAL)

    for step in range(1, num_steps + 1):
        particles[:, 0] += particles[:, 1] * DT
        particles[:, 0] %= LX

        step_accepted_collisions, step_selected_pairs, step_sum_separation = 0.0, 0.0, 0.0
        step_prob_exceed = 0.0
        duplicate_check_array = np.full(TOTAL_PARTICLES_SIM, -1, dtype=np.int64)
        cell_indices = (particles[:, 0] / cell_width).astype(np.int64)

        for i in range(NUM_CELLS_X):
            indices_in_cell_i = np.where(cell_indices == i)[0]
            n_acc, new_sigma_g_max, n_sel, sum_sep, n_prob_exceed = perform_collisions(
                method_code, particles, LX, indices_in_cell_i,
                CELL_VOLUME_CONCEPTUAL, DT, FNUM, rng_state,
                sigma_g_max[i], n_sel_fraction, duplicate_check_array
            )
            sigma_g_max[i] = new_sigma_g_max
            step_accepted_collisions += n_acc
            step_selected_pairs += n_sel
            step_sum_separation += sum_sep
            step_prob_exceed += n_prob_exceed

        total_accepted_collisions += step_accepted_collisions
        total_selected_pairs += step_selected_pairs
        total_collision_separation += step_sum_separation
        total_prob_exceed_collisions += step_prob_exceed

        if step % SAMPLING_INTERVAL == 0 and step > MTC:
            current_time = step * DT

            # Memory-efficient history collection with rolling buffer
            if history_sample_count < MAX_HISTORY_SAMPLES:
                time_history.append(current_time)
                history_sample_count += 1
            else:
                # Use rolling buffer - remove oldest, add newest
                time_history.pop(0)
                time_history.append(current_time)

            collisions_this_interval = total_accepted_collisions - last_sampled_total_collisions
            prob_exceed_this_interval = total_prob_exceed_collisions - last_sampled_prob_exceed

            # Calculate probability exceed ratio
            prob_exceed_ratio = (prob_exceed_this_interval / collisions_this_interval) if collisions_this_interval > 0 else 0.0

            # Apply same rolling buffer logic to all history lists
            if len(freq_ratio_history) >= MAX_HISTORY_SAMPLES:
                freq_ratio_history.pop(0)
                sof_history.pop(0)
                prob_exceed_history.pop(0)

            time_this_interval = SAMPLING_INTERVAL * DT
            current_temp = (0.5 * MASS_AR / (1.5 * KB)) * np.mean(np.sum(particles[:,1:4]**2, axis=1))

            cf_numerical = calculate_numerical_collision_frequency(collisions_this_interval, TOTAL_PARTICLES_SIM, time_this_interval)
            cf_theoretical = calculate_theoretical_collision_frequency(current_temp, N_DENSITY_REAL)
            ratio = cf_numerical / cf_theoretical if cf_theoretical > 0 else 0
            freq_ratio_history.append(ratio)
            prob_exceed_history.append(prob_exceed_ratio)
            last_sampled_total_collisions = total_accepted_collisions
            last_sampled_prob_exceed = total_prob_exceed_collisions

            sof_interval = (step_sum_separation / step_accepted_collisions) / mfp if step_accepted_collisions > 0 and mfp > 0 else 0
            sof_history.append(sof_interval)

            cell_counts = np.bincount(cell_indices, minlength=NUM_CELLS_X)
            deviations = cell_counts - int(navg_theory)
            for dev in deviations:
                if -MAXD <= dev <= MAXD:
                    density_fluctuation_hist[dev + MAXD] += 1

            current_cell_props = np.zeros((5, NUM_CELLS_X))
            for i in range(NUM_CELLS_X):
                indices = np.where(cell_indices == i)[0]
                N_i = len(indices)
                current_cell_props[0, i] = N_i
                if N_i > 1:
                    v_avg = np.mean(particles[indices, 1:4], axis=0)
                    current_cell_props[1:4, i] = v_avg
                    v_var = np.var(particles[indices, 1:4])
                    temp_i = (MASS_AR / (3 * KB)) * v_var * 3
                    current_cell_props[4, i] = temp_i

            # Multi-cell temporal correlation calculation
            for cell_idx, cell_num in enumerate(cells_to_analyze):
                temporal_samples_storage_multi[cell_idx, :, :-1] = temporal_samples_storage_multi[cell_idx, :, 1:]
                temporal_samples_storage_multi[cell_idx, :, -1] = current_cell_props[:, cell_num]

            nsmpt_multi += 1
            # Only start computing correlations after buffer is full
            if nsmpt_multi > MTC:
                for cell_idx, cell_num in enumerate(cells_to_analyze):
                    center_values_t = temporal_samples_storage_multi[cell_idx, :, MTC]
                    # Calculate running mean for each property across time buffer
                    current_means = np.mean(temporal_samples_storage_multi[cell_idx, :, :], axis=1)

                    for lag in range(2 * MTC + 1):
                        past_values = temporal_samples_storage_multi[cell_idx, :, lag]
                        # Compute covariance properly: (X - mean_X) * (Y - mean_Y)
                        correlation_products = (center_values_t - current_means) * (past_values - current_means)
                        temporal_corr_sums_multi[cell_idx, :, lag] += correlation_products

            # Fixed spatial correlation calculation
            nsmpx += 1
            center_props_s = current_cell_props[:, CENTRAL_CELL_IDX]
            # Calculate spatial means across all cells for normalization
            spatial_means = np.mean(current_cell_props, axis=1)

            for lag in range(-MXC, MXC + 1):
                cell_idx_to_compare = (CENTRAL_CELL_IDX + lag + NUM_CELLS_X) % NUM_CELLS_X
                compare_props = current_cell_props[:, cell_idx_to_compare]
                # Proper spatial correlation: (X_center - mean) * (X_lag - mean)
                correlation_products = (center_props_s - spatial_means) * (compare_props - spatial_means)
                spatial_corr_sums[:, lag + MXC] += correlation_products

        # Memory-efficient speed sampling with limits
        if step >= speed_sampling_start_step and step % (SAMPLING_INTERVAL * 2) == 0:
            if len(sampled_speeds_accumulator) < MAX_SPEED_SAMPLES:
                # Sample subset of particles to avoid memory explosion
                sample_size = min(1000, TOTAL_PARTICLES_SIM)  # Sample max 1000 particles per interval
                # Use simple random sampling instead of rng_state.choice
                sample_indices = np.random.choice(TOTAL_PARTICLES_SIM, size=sample_size, replace=False)
                particle_speeds = np.sqrt(np.sum(particles[sample_indices, 1:4]**2, axis=1))
                sampled_speeds_accumulator.extend(particle_speeds)
            # Stop collecting when limit reached to prevent memory overflow

        if step % (num_steps // 10) == 0:
            progress = step / num_steps * 100
            current_time_ns = step * DT * 1e9

            # Memory usage estimation
            history_size = len(time_history) * 4  # 4 history lists
            speed_size = len(sampled_speeds_accumulator)
            total_samples = history_size + speed_size

            print(f"Step: {step}/{num_steps} ({progress:.1f}%) - Time: {current_time_ns:.1f} ns")
            print(f"  Data samples: {total_samples:,} (History: {history_size:,}, Speeds: {speed_size:,})")

            # Garbage collection hint to help with memory management
            import gc
            gc.collect()

    end_time = time.time()
    print(f"\nSimulation finished in {end_time - start_time:.2f} seconds.")

    # Final memory usage summary
    final_history_size = len(time_history) * 4
    final_speed_size = len(sampled_speeds_accumulator)
    print(f"Final data collection: {final_history_size:,} history samples, {final_speed_size:,} speed samples")

    # Memory cleanup
    gc.collect()

    final_temp = (0.5 * MASS_AR / (1.5 * KB)) * np.mean(np.sum(particles[:,1:4]**2, axis=1))
    print(f"\nFinal Equilibrium Temperature: {final_temp:.2f} K")

    acceptance_ratio = total_accepted_collisions / total_selected_pairs if total_selected_pairs > 0 else 0
    mean_separation = total_collision_separation / total_accepted_collisions if total_accepted_collisions > 0 else 0
    sof = mean_separation / mfp if mfp > 0 else 0
    overall_prob_exceed_ratio = total_prob_exceed_collisions / total_accepted_collisions if total_accepted_collisions > 0 else 0

    # Calculate dx/dt according to the reference
    cm = np.sqrt(2 * KB * T_INIT / MASS_AR)  # Most probable thermal velocity
    tc = mfp / cm  # Mean collision time
    dx_dimensionless = cell_width / mfp  # Dimensionless cell size
    dt_dimensionless = DT / tc  # Dimensionless time step
    dx_dt_ratio = dx_dimensionless / dt_dimensionless

    print(f"Total Collisions Accepted: {int(total_accepted_collisions)}")
    print(f"Total Pairs Selected: {int(total_selected_pairs)}")
    print(f"Collision Acceptance Ratio: {acceptance_ratio:.4f}")
    print(f"Mean Free Path (MFP): {mfp:.4e} m")
    print(f"Mean Collision Separation / MFP (SoF): {sof:.4f}")
    print(f"Overall Probability Exceed Ratio: {overall_prob_exceed_ratio:.4f}")
    print(f"dx/dt (normalized): {dx_dt_ratio:.4f}")

    # Memory usage note
    if len(sampled_speeds_accumulator) >= MAX_SPEED_SAMPLES:
        print(f"Note: Speed sampling was limited to {MAX_SPEED_SAMPLES:,} samples to prevent memory overflow")
    if len(time_history) >= MAX_HISTORY_SAMPLES:
        print(f"Note: History tracking used rolling buffer limited to {MAX_HISTORY_SAMPLES:,} samples")

    # Report N_sel for GBT and SGBT methods
    if method_code in [4, 6]:
        requested_n_sel = int(n_sel_fraction)
        if requested_n_sel >= PARTICLES_PER_CELL_INIT - 1:
            print(f"N_sel requested: {requested_n_sel} (>= {PARTICLES_PER_CELL_INIT - 1}), used Standard SBT instead")
        else:
            actual_n_sel = requested_n_sel
            print(f"N_sel used: {actual_n_sel} (per cell with {PARTICLES_PER_CELL_INIT} particles)")

    # Calculate mean free path for plotting
    sigma_ref = PI * D_REF_AR**2
    mfp = 1 / (np.sqrt(2) * sigma_ref * N_DENSITY_REAL)

    if sampled_speeds_accumulator:
        plot_results(np.array(sampled_speeds_accumulator), final_temp, collision_method_name,
                    freq_ratio_history, time_history, sof_history, acceptance_ratio, mfp, prob_exceed_history)

    plot_advanced_stats(density_fluctuation_hist, temporal_corr_sums_multi, spatial_corr_sums,
                       nsmpt_multi, nsmpx, navg_theory, collision_method_name, DT, cell_width, cells_to_analyze)

if __name__ == "__main__":
    # Get initial parameters from user
    print("=== DSMC Relaxation Simulation Setup ===")

    # Get particles per cell from user
    while True:
        try:
            particles_input = int(input("Enter initial number of particles per cell (e.g., 10): "))
            if particles_input >= 2:
                break
            else:
                print("Invalid input. Please enter a value >= 2.")
        except ValueError:
            print("Invalid input. Please enter an integer.")

    # Get collision method from user
    while True:
        choice = input("Choose method: [1] SBT, [2] DCP, [3] NTC, [4] GBT, [5] SSBT, [6] SGBT, [7] MFS, [8] NN: ")
        if choice in ['1', '2', '3', '4', '5', '6', '7', '8']:
            method_code = int(choice)
            n_sel_fraction = 0.5  # Default for non-GBT/SGBT methods
            n_sel_for_params = None  # For parameter calculation

            # Get N_sel for GBT and SGBT methods
            if choice in ['4', '6']:
                method_name = {'4': 'GBT', '6': 'SGBT'}[choice]
                while True:
                    try:
                        n_sel_input = int(input(f"Enter the N_sel value for {method_name} (e.g., 2 for selecting 2 particles per cell): "))
                        if n_sel_input >= 1:
                            n_sel_fraction = n_sel_input  # Store the actual N_sel value
                            n_sel_for_params = n_sel_input  # For parameter calculation
                            break
                        else:
                            print("Invalid value. N_sel must be >= 1.")
                    except ValueError:
                        print("Invalid input. Please enter an integer.")

            # Calculate and report simulation parameters AFTER getting both PPC and N_sel
            calculate_simulation_parameters(particles_input, n_sel_for_params)

            # Run the simulation
            run_dsmc_simulation(method_code=method_code, n_sel_fraction=n_sel_fraction)
            break
        else:
            print("Invalid input.")